# Update 4 — Uncertainty-budget audit

**Revision note (v2.0):** The Supplemental-Material uncertainty table in the submitted version contained entries that could not be reproduced from the code:

| Old entry | Actual origin | Problem |
|---|---|---|
| Finite sample & serial correlation 1.69×10⁻¹⁰ | mean of three HAC (lag=3) standard errors | statistically meaningless |
| Temperature sensor calibration 1.42×10⁻¹⁰ | EIV std with input 0.5 °C | spec is ±1.0 °C (2× understated) |
| Humidity sensor calibration 1.25×10⁻¹⁰ | EIV std with input 3.0 % | wrong magnitude |
| Pressure sensor calibration 9.11×10⁻¹³ | EIV std with input 12 Pa = 0.12 hPa | spec is ±1 hPa (8.3× understated) |

This notebook rebuilds the budget from first principles:

1. **Statistical uncertainties:** moving-block bootstrap over physical-time blocks (duration swept 0.56--48 h, $B = 5000$, coverage reported per duration), cross-checked against HAC with lag = 156 (lag = 3 underestimates by ≈5×). The fixed row-block length $L = 156$ is withdrawn.
2. **Systematic uncertainties:** sensor *gain* errors from BME280 specifications (constant offsets are absorbed by the intercept $n_0$ and do not affect the coefficients).
3. **Two-measurand structure:** Table S1a (single-epoch $n$), Table S1b (coefficients).

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf
import sys, os, time
sys.path.append('..')
from models.mathar.Mathar2007 import n as n_mathar_scalar

df = pd.read_csv('../../data/processed/full_data.csv')
df = df.sort_values('time').reset_index(drop=True)

T_C, H_pct, P_hPa = df['temperature'].values, df['humidity'].values, df['pressure'].values
n_data = df['n_1762'].values

# Mathar cloud for residual decomposition (Table S1a)
lam_um, T_K, P_Pa = 1.762, T_C + 273.15, P_hPa * 100.0
n_mathar = np.array([n_mathar_scalar(lam_um, Tk, pp, hh) for Tk, pp, hh in zip(T_K, P_Pa, H_pct)])

X = np.column_stack([np.ones(len(df)), T_C, H_pct, P_hPa])   # P in hPa -> coefficients in hPa^-1
N = len(df)
print(f"N = {N}")

N = 145784


In [2]:
# Statistical SE over PHYSICAL-TIME blocks; the duration is swept and the
# coverage fraction of each pool is reported. Supersedes the fixed L = 156 row
# blocks, which could straddle the 111-day gap and understated the spread.
from physical_blocks import (D_LIST_HOURS, build_blocks, coefficient_bootstrap,
                             coverage, segment_bounds, time_seconds)

t_sec = time_seconds(df['time'])
seg = segment_bounds(t_sec)
D_SWEEP = D_LIST_HOURS   # 0.559 / 7.167 / 24 / 48 h, from D_LIST_SECONDS

def physical_block_se(D_hours, B=5000, seed=42):
    blocks = build_blocks(t_sec, D_hours * 3600.0, seg)
    draws = coefficient_bootstrap(X, n_data, blocks, n_draws=B, seed=seed)
    return draws.std(axis=0), coverage(blocks, len(df)), len(blocks)

t0 = time.time()
names = ['n0', 'aT', 'aH', 'aP']
se_by_D = {}
for D in D_SWEEP:
    se, cov, nbl = physical_block_se(D)
    se_by_D[D] = se
    print(f'D = {D:6.2f} h | {nbl:5d} blocks | coverage {cov:.3f} | '
          f'SE aT {se[1]:.3e}  SE aH {se[2]:.3e}  SE aP {se[3]:.3e}')
print(f'Bootstrap sweep done in {time.time()-t0:.0f} s')

# Descriptive-only alias: the shortest duration. No single nominal value is
# adopted; downstream cells use the range across D_SWEEP.
se_boot = se_by_D[D_SWEEP[0]]

# HAC (Newey-West) with lag = 156 (Bartlett), cross-check
model_ols = sm.OLS(n_data, X).fit()
hac156 = sm.OLS(n_data, X).fit(cov_type='HAC', cov_kwds={'maxlags': 156})
hac3   = sm.OLS(n_data, X).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
print()
print('HAC (lag=156) vs the physical-time block range:')
for j in range(1, 4):
    lo = min(se_by_D[D][j] for D in D_SWEEP)
    hi = max(se_by_D[D][j] for D in D_SWEEP)
    print(f'  {names[j]}: HAC156={hac156.bse[j]:.6e}  blocks [{lo:.3e}, {hi:.3e}]')
print()
print('HAC lag=3 underestimation factors:')
for j in range(1, 4):
    print(f'  {names[j]}: HAC156/HAC3 = {hac156.bse[j]/hac3.bse[j]:.2f}x')


D =   0.56 h |  1379 blocks | coverage 0.996 | SE aT 1.143e-09  SE aH 1.010e-09  SE aP 7.949e-10
D =   7.17 h |   122 blocks | coverage 0.936 | SE aT 3.379e-09  SE aH 3.146e-09  SE aP 2.399e-09
D =  24.00 h |    31 blocks | coverage 0.580 | SE aT 4.796e-09  SE aH 4.480e-09  SE aP 4.619e-09
D =  48.00 h |     5 blocks | coverage 0.150 | SE aT 1.812e-08  SE aH 1.210e-08  SE aP 8.180e-09
Bootstrap sweep done in 1 s

HAC (lag=156) vs the physical-time block range:
  aT: HAC156=1.407200e-09  blocks [1.143e-09, 1.812e-08]
  aH: HAC156=1.157710e-09  blocks [1.010e-09, 1.210e-08]
  aP: HAC156=9.686824e-10  blocks [7.949e-10, 8.180e-09]

HAC lag=3 underestimation factors:
  aT: HAC156/HAC3 = 5.03x
  aH: HAC156/HAC3 = 5.16x
  aP: HAC156/HAC3 = 5.20x


In [3]:
# ---------------------------------------------------------------------
# 3. Deterministic sensor gain bounds, reported SEPARATELY from statistics
#    (offset absorbed by n0; gain propagates as alpha_meas = alpha_true / g)
#    u_syst / u_tot / U(k=2) are RETRACTED.  Merging the paired-difference
#    sensitivity of nb09 with the measured coefficient's statistical
#    uncertainty requires a specified and propagated joint probability
#    model, which is not available.  Deterministic envelopes and standard
#    uncertainties are therefore reported separately and never added.
# ---------------------------------------------------------------------
alpha_T = model_ols.params[1]
alpha_H = model_ols.params[2]
alpha_P = model_ols.params[3]          # hPa^-1 (design matrix uses hPa)

# campaign spans (observed)
span_T = (T_C.max() - T_C.min())
span_H = (H_pct.max() - H_pct.min())
span_P = (P_hPa.max() - P_hPa.min())
print(f'observed spans: T {span_T:.2f} K, H {span_H:.2f} %RH, P {span_P:.2f} hPa')

# worst-case gain error from manufacturer accuracy over the campaign ranges
# temperature:  +/-1.0 degC ; humidity: +/-3 %RH at 25 degC, degrading to ~4 %RH
#               over the campaign temperature range (incl. drift);
# pressure:     +/-1 hPa
acc_T, acc_H, acc_P = 1.0, 4.0, 1.0   # degC, %RH, hPa

gain_err_T = 2*acc_T/span_T            # worst-case gain deviation
gain_err_H = 2*acc_H/span_H
gain_err_P = 2*acc_P/span_P

# single-branch endpoint: coefficient change for the full worst-case gain
# deviation.  DETERMINISTIC and listed for reference only; it rescales one
# coefficient in isolation and omits the Ciddor-anchor and Mathar branches.
end_T = gain_err_T * abs(alpha_T)
end_H = gain_err_H * abs(alpha_H)
end_P = gain_err_P * abs(alpha_P)

print()
print('Single-branch deterministic endpoints (NOT standard uncertainties):')
print(f'  alpha_T: gain {100*gain_err_T:.1f}%, endpoint {end_T:.3e}')
print(f'  alpha_H: gain {100*gain_err_H:.1f}%, endpoint {end_H:.3e}')
print(f'  alpha_P: gain {100*gain_err_P:.1f}%, endpoint {end_P:.3e}')

# ---------------------------------------------------------------------
# 4. Table S1b: values, deterministic endpoints, statistical SEs
# ---------------------------------------------------------------------
print()
print('Table S1b  --  coefficients, deterministic endpoints, statistical SEs')
print('='*110)
print(f"{'Coeff':<16}{'Value':>16}{'Endpoint(det)':>16}{'HAC156':>14}{'SE blocks min':>16}{'SE blocks max':>16}")
vals = [
    ('alpha_T (K^-1)',  alpha_T, end_T, hac156.bse[1], 1),
    ('alpha_H (%RH^-1)', alpha_H, end_H, hac156.bse[2], 2),
    ('alpha_P (hPa^-1)', alpha_P, end_P, hac156.bse[3], 3),
]
for name, v, end, uh, j in vals:
    lo = min(se_by_D[D][j] for D in D_SWEEP)
    hi = max(se_by_D[D][j] for D in D_SWEEP)
    print(f"{name:<16}{v:>16.6e}{end:>16.3e}{uh:>14.3e}{lo:>16.3e}{hi:>16.3e}")
print()
print('No combined standard uncertainty is quoted for the coefficients')
print('pending a measurand-specific propagation; the deterministic endpoint')
print('and the statistical SE are different quantities and are not added.')


observed spans: T 17.10 K, H 26.38 %RH, P 42.17 hPa

Single-branch deterministic endpoints (NOT standard uncertainties):
  alpha_T: gain 11.7%, endpoint 1.035e-07
  alpha_H: gain 30.3%, endpoint 3.988e-09
  alpha_P: gain 4.7%, endpoint 1.231e-08

Table S1b  --  coefficients, deterministic endpoints, statistical SEs
Coeff                      Value   Endpoint(det)        HAC156   SE blocks min   SE blocks max
alpha_T (K^-1)     -8.847425e-07       1.035e-07     1.407e-09       1.143e-09       1.812e-08
alpha_H (%RH^-1)   -1.315242e-08       3.988e-09     1.158e-09       1.010e-09       1.210e-08
alpha_P (hPa^-1)    2.594906e-07       1.231e-08     9.687e-10       7.949e-10       8.180e-09

No combined standard uncertainty is quoted for the coefficients
pending a measurand-specific propagation; the deterministic endpoint
and the statistical SE are different quantities and are not added.


In [4]:
# single-epoch budget (Table S1a) incl. residual decomposition
resid_data   = n_data - X @ model_ols.params
sigma_n      = np.std(resid_data, ddof=0)

# Mathar surrogate fit on identical rows -> model nonlinearity component
beta_m, *_ = np.linalg.lstsq(X, n_mathar, rcond=None)
sigma_model = np.std(n_mathar - X @ beta_m, ddof=0)
sigma_noise = np.sqrt(sigma_n**2 - sigma_model**2)

print(f"sigma_n (residual scatter)        = {sigma_n:.6e}")
print(f"  of which model nonlinearity     = {sigma_model:.6e}")
print(f"  of which measurement noise      = {sigma_noise:.6e}")
print(f"check: sqrt(noise^2+model^2)      = {np.sqrt(sigma_noise**2+sigma_model**2):.6e}")
print()
print("Table S1a  —  single-epoch refractive index budget")
print("="*70)
rows = [
    ("Residual scatter (incl. spatial/temporal mismatch)", sigma_n),
    ("780 nm frequency reference",                         1.0e-9),
    ("1762 nm frequency reference",                        0.0),   # <1e-12
]
tot = np.sqrt(sum(r[1]**2 for r in rows))
for name, v in rows:
    print(f"  {name:<55} {v:.3e}")
print(f"  {'Combined standard uncertainty':<55} {tot:.3e}")
print(f"  {'Combined expanded (k=2)':<55} {2*tot:.3e}")
print()
print("Residual decomposition (informational, not part of the RSS):")
print(f"  measurement noise  = {sigma_noise:.3e}")
print(f"  model nonlinearity = {sigma_model:.3e}")
print(f"  check sqrt(n^2+m^2)= {np.sqrt(sigma_noise**2+sigma_model**2):.3e}")

# ---------------------------------------------------------------------
# 6. Humidity channel against the deterministic endpoint
#    u_syst / u_tot / U(k=2) are RETRACTED for the coefficients (see cell 3):
#    the deterministic envelope and the statistical component are reported
#    separately and no combined standard uncertainty is formed.
# ---------------------------------------------------------------------
dAlphaH = model_ols.params[2] - beta_m[2]     # +4.33e-9 (full campaign)
lo_T = min(se_by_D[D][1] for D in D_SWEEP)
hi_T = max(se_by_D[D][1] for D in D_SWEEP)
lo_P = min(se_by_D[D][3] for D in D_SWEEP)
hi_P = max(se_by_D[D][3] for D in D_SWEEP)
print()
print('Delta-alpha (full campaign); statistical SE range over physical blocks:')
print(f"  Delta_aT = {model_ols.params[1]-beta_m[1]:+.4e}  SE [{lo_T:.2e}, {hi_T:.2e}]")
print(f"  Delta_aP = {model_ols.params[3]-beta_m[3]:+.4e}  SE [{lo_P:.2e}, {hi_P:.2e}]")
print()
print('Humidity channel:')
print(f"  Delta_aH        = {dAlphaH:+.4e}")
print(f"  endpoint (det.) = {end_H:.4e}   (gain {100*gain_err_H:.1f}%)")
resid_beyond = abs(dAlphaH) - end_H
print(f"  beyond endpoint = {resid_beyond:+.2e}  -> the single-branch endpoint alone does not span it")
print(f"  no u_comb is formed: u_syst / u_tot / U(k=2) are retracted (see cell 3)")


sigma_n (residual scatter)        = 1.836658e-07
  of which model nonlinearity     = 4.430796e-08
  of which measurement noise      = 1.782412e-07
check: sqrt(noise^2+model^2)      = 1.836658e-07

Table S1a  —  single-epoch refractive index budget
  Residual scatter (incl. spatial/temporal mismatch)      1.837e-07
  780 nm frequency reference                              1.000e-09
  1762 nm frequency reference                             0.000e+00
  Combined standard uncertainty                           1.837e-07
  Combined expanded (k=2)                                 3.673e-07

Residual decomposition (informational, not part of the RSS):
  measurement noise  = 1.782e-07
  model nonlinearity = 4.431e-08
  check sqrt(n^2+m^2)= 1.837e-07

Delta-alpha (full campaign); statistical SE range over physical blocks:
  Delta_aT = -3.1062e-09  SE [1.14e-09, 1.81e-08]
  Delta_aP = +2.3946e-09  SE [7.95e-10, 8.18e-09]

Humidity channel:
  Delta_aH        = +4.3334e-09
  endpoint (det.) = 3.9878e

- **Bootstrap SE over physical-time blocks (duration swept 0.56-48 h, coverage 0.996-0.150):** $\alpha_T$: at the shortest duration 1.14e-9 (aT), 1.01e-9 (aH) and 7.95e-10 (aP); the SE grows with the block duration. HAC (lag 156) lies inside the swept block range; HAC (lag 3) underestimates by about 5x. No single nominal block duration is adopted.
- **Single-branch deterministic endpoints (gain bounds; reported separately from the statistical SE):**
  - $\alpha_T$: gain bound 11.7 % (span 17.1 K), endpoint $1.03\times10^{-7}$
  - $\alpha_H$: gain bound 30.3 % ($\pm$ 3 %RH at 25 °C degrading to ≈$\pm$ 4 %RH over the campaign range, span 26.4 %), endpoint $3.99\times10^{-9}$
  - $\alpha_P$: gain bound 4.7 % (span 42.2 hPa), endpoint $1.23\times10^{-8}$
- **Humidity significance:** $\Delta\alpha_H = +4.33\times10^{-9}$ exceeds the endpoint $3.99\times10^{-9}$ by $0.34\times10^{-9}$. The single-branch endpoint does not span the difference once the deterministic envelope is set alongside it; no combined ratio is quoted, because u_syst, u_tot and U(k=2) are retracted and the paired-bootstrap SE of the difference is block-duration dependent. The difference is reported as an unresolved systematic.
- **Table S1a:** combined expanded uncertainty ($k=2$) $3.68\times10^{-7}$, dominated by measurement noise $1.78\times10^{-7}$ (model nonlinearity only $4.4\times10^{-8}$). The residual decomposition rows are informational; the RSS combination uses the residual scatter only.